# SatQuery: 1,000-area CROMA pipeline
Run the cells **one at a time** and inspect each completion report before continuing.
Use a free T4 GPU runtime (Runtime → Change runtime type). This notebook uses the previously downloaded `MyDrive/SatQuery/bigearthnet-v2-1000` ZIPs and saves separate stages under `MyDrive/SatQuery/pipeline-1000`.

The official area splits remain 600 train / 200 validation / 200 test. All 225 tokens from an area stay together. Existing completed exports are verified and reused. An existing model is reused; choose a new output folder for a new experiment.

Coverage percentages are **not confidence**. Final inference attaches held-out class errors and support. Label matching verifies CLC reference-map geometry, not independent ground truth at 80 metres.


## Install the included, checksum-verified package

In [ ]:
import base64, hashlib, subprocess, sys
from pathlib import Path
wheel_path = Path('/content/satquery_preprocessing-0.1.0-py3-none-any.whl')
wheel_bytes = base64.b64decode(
    '...'
)
assert hashlib.sha256(wheel_bytes).hexdigest() == 'b0b83b2a5eb617f87dec13241aff0c9e5fde703208c4f07e99a93acc6336e559'
wheel_path.write_bytes(wheel_bytes)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade', '--force-reinstall', '--no-deps', str(wheel_path)])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', str(wheel_path)+'[downloads,features]'])
print('Verified SatQuery package installed')


## Connect the previously approved Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 1. Convert images into verified CROMA tensors

In [ ]:
# Stage 1 — verified BigEarthNet ZIPs to CROMA input tensors
import json
import shutil
import stat
import zipfile
from collections import Counter
from pathlib import Path

import pandas as pd
import torch

from satquery.prepare_croma import export_inputs
from satquery.preprocessing import DEFAULT_PROFILE, sha256

SOURCE = Path('/content/drive/MyDrive/SatQuery/bigearthnet-v2-1000')
PIPELINE = Path('/content/drive/MyDrive/SatQuery/pipeline-1000')
RAW = Path('/content/satquery-pipeline-raw')
PREPARED = PIPELINE / 'prepared'
assert SOURCE.is_dir(), 'Connect the Google Drive containing the completed download first'
PIPELINE.mkdir(parents=True, exist_ok=True)
torch.set_num_threads(2)
report = json.loads((SOURCE / 'download.json').read_text())
assert report['complete'] and report['sample_count'] == 1000
assert sha256(SOURCE / 'metadata.parquet') == report['metadata_sha256']
assert sha256(SOURCE / 'selection.json') == report['selection_sha256']
metadata = pd.read_parquet(SOURCE / 'metadata.parquet')
assert not metadata.patch_id.duplicated().any() and not metadata.s1_name.duplicated().any()
assert metadata.split.value_counts().to_dict() == {'train': 600, 'validation': 200, 'test': 200}

if not PREPARED.exists():
    RAW.mkdir(parents=True, exist_ok=True)
    for shard in report['shards']:
        archive = SOURCE / shard['archive']
        assert sha256(archive) == shard['sha256'], archive.name
        with zipfile.ZipFile(archive) as zipped:
            for member in zipped.infolist():
                path = (RAW / member.filename).resolve()
                assert RAW.resolve() in path.parents, 'Unsafe ZIP path'
                assert not stat.S_ISLNK(member.external_attr >> 16), 'Unexpected ZIP symlink'
            zipped.extractall(RAW)
        print('Verified and extracted', archive.name, flush=True)
    # Each ZIP has its own subset metadata. Restore the combined index last.
    shutil.copyfile(SOURCE / 'metadata.parquet', RAW / 'metadata.parquet')
    print('Converting 1,000 areas; validating native grids, band order and masks...', flush=True)
    export_inputs(RAW, PREPARED, batch_size=32, progress=lambda s: print(s, flush=True))

manifest = json.loads((PREPARED / 'manifest.json').read_text())
validation = json.loads((PREPARED / 'validation.json').read_text())
assert validation['passed'] and sha256(PREPARED / 'manifest.json') == validation['manifest_sha256']
assert manifest['sample_count'] == 1000 and manifest['metadata_sha256'] == report['metadata_sha256']
assert manifest['channel_profile']['optical'] == list(DEFAULT_PROFILE.optical)
assert manifest['channel_profile']['sar'] == list(DEFAULT_PROFILE.sar)
ids, splits = [], Counter()
for batch in manifest['batches']:
    folder = PREPARED / batch['directory']
    for name, digest in batch['sha256'].items():
        assert sha256(folder / name) == digest, str(folder / name)
    info = json.loads((folder / 'batch.json').read_text())
    n = batch['sample_count']
    tensors = torch.load(folder / 'croma_inputs.pt', map_location='cpu', weights_only=True)
    for key, channels in [('optical_images', 12), ('SAR_images', 2)]:
        value = tensors[key]
        assert value.shape == (n, channels, 120, 120) and value.dtype == torch.float32
        assert torch.isfinite(value).all() and value.min() >= 0 and value.max() <= 1
    maps = torch.load(folder / 'reference_maps.pt', map_location='cpu', weights_only=True)
    assert maps.shape == (n,120,120) and maps.dtype == torch.int64
    ids.extend(s['patch_id'] for s in info['samples'])
    splits.update(s['split'] for s in info['samples'])
    del tensors, maps
assert ids == metadata.patch_id.tolist() and dict(splits) == report['counts']
stage_report = {
    'stage': 1, 'passed': True, 'sample_count': len(ids), 'splits': dict(splits),
    'optical_shape': [len(ids),12,120,120], 'sar_shape': [len(ids),2,120,120],
    'reference_map_shape': [len(ids),120,120], 'stored_batches': len(manifest['batches']),
    'prepared_manifest_sha256': sha256(PREPARED / 'manifest.json'),
    'download_report_sha256': sha256(SOURCE / 'download.json'),
    'output': str(PREPARED), 'excluded_areas': [],
}
(PIPELINE / 'stage-1-report.json').write_text(json.dumps(stage_report, indent=2) + '\n')
print('STAGE 1 COMPLETE:', json.dumps(stage_report), flush=True)


## 2. Extract frozen CROMA features

In [ ]:
# Stage 2 — frozen CROMA feature extraction
import json
from pathlib import Path
import torch
from satquery.features import CromaFeatureExtractor, extract_features
from satquery.preprocessing import sha256

PIPELINE = Path('/content/drive/MyDrive/SatQuery/pipeline-1000')
PREPARED, FEATURES = PIPELINE / 'prepared', PIPELINE / 'features'
stage1 = json.loads((PIPELINE / 'stage-1-report.json').read_text())
assert stage1['passed'] and stage1['sample_count'] == 1000
assert sha256(PREPARED / 'manifest.json') == stage1['prepared_manifest_sha256']
torch.set_num_threads(2)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('CROMA device:', device, flush=True)
if not FEATURES.exists():
    extractor = CromaFeatureExtractor.from_pretrained(cache_dir=PIPELINE / 'croma-checkpoint', device=device)
    extract_features(PREPARED, FEATURES, extractor, batch_size=8 if device == 'cuda' else 2,
                     progress=lambda s: print(s, flush=True))
    del extractor
    if device == 'cuda': torch.cuda.empty_cache()
manifest = json.loads((FEATURES / 'manifest.json').read_text())
validation = json.loads((FEATURES / 'validation.json').read_text())
assert validation['passed'] and manifest['sample_count'] == 1000
assert sha256(FEATURES / 'manifest.json') == validation['manifest_sha256']
assert manifest['source_manifest_sha256'] == stage1['prepared_manifest_sha256']
for batch in manifest['batches']:
    for name, digest in batch['sha256'].items():
        assert sha256(FEATURES / batch['directory'] / name) == digest
stage2 = {'stage':2,'passed':True,'sample_count':1000,'tokens_per_area':225,
          'feature_dimension':768,'spatial_features':['optical_encodings','SAR_encodings','joint_encodings'],
          'feature_manifest_sha256':sha256(FEATURES / 'manifest.json'),'device':manifest['device'],
          'checkpoint_sha256':manifest['model']['checkpoint_sha256'],'output':str(FEATURES)}
(PIPELINE / 'stage-2-report.json').write_text(json.dumps(stage2, indent=2)+'\n')
print('STAGE 2 COMPLETE:', json.dumps(stage2), flush=True)


## 3–4. Generate class percentages and independently audit every 80 m block

In [ ]:
# Stages 3–4: coverage targets and independent exhaustive spatial audit.
import json
import zipfile
from collections import Counter
from pathlib import Path

import numpy as np
import rasterio
import torch
from rasterio.io import MemoryFile
from rasterio.windows import Window, bounds

from satquery.preprocessing import sha256
from satquery.targets import export_targets

P = Path('/content/drive/MyDrive/SatQuery/pipeline-1000')
SOURCE = P.parent / 'bigearthnet-v2-1000'
prepared, features, targets = P/'prepared', P/'features', P/'targets'
previous = json.loads((P/'stage-2-report.json').read_text())
assert previous['passed'] and sha256(features/'manifest.json') == previous['feature_manifest_sha256']
torch.set_num_threads(2)
if not targets.exists():
    export_targets(prepared, features, targets, progress=lambda s: print(s, flush=True))
pm, fm, tm = [json.loads((folder/'manifest.json').read_text()) for folder in (prepared,features,targets)]
validation = json.loads((targets/'validation.json').read_text())
assert validation['passed'] and validation['manifest_sha256'] == sha256(targets/'manifest.json')
assert tm['feature_manifest_sha256'] == sha256(features/'manifest.json')
assert tm['prepared_manifest_sha256'] == sha256(prepared/'manifest.json')

# Independent transcription of official v2 Table 1. Do not use the target module's lookup.
groups = [(111,112),(121,),(211,212,213),(221,222,223,241),(231,),(242,),(243,),
          (244,),(311,),(312,),(313,),(321,333),(322,323),(324,),(331,),(411,412),
          (421,422),(511,512),(521,522,523)]
excluded = {0,999,122,123,124,131,132,133,141,142,332,334,335,423}
lookup = {code:i for i,group in enumerate(groups) for code in group}
assert [c['clc_codes'] for c in tm['classes']] == [list(g) for g in groups]
original = {}
download = json.loads((SOURCE/'download.json').read_text())
for shard in download['shards']:
    archive = SOURCE/shard['archive']
    assert sha256(archive) == shard['sha256']
    with zipfile.ZipFile(archive) as z:
        for name in z.namelist():
            if name.endswith('_reference_map.tif'):
                patch = Path(name).name.removesuffix('_reference_map.tif')
                assert patch not in original
                with MemoryFile(z.read(name)) as memory, memory.open() as raster:
                    original[patch] = (raster.read(1), raster.transform, raster.crs, tuple(raster.bounds))
print('Read all original reference TIFFs:',len(original),flush=True)
assert len(original) == 1000
stats = {s:Counter() for s in ('train','validation','test')}
class_tokens = {s:np.zeros(19,dtype=np.int64) for s in stats}
class_pixels = {s:np.zeros(19,dtype=np.int64) for s in stats}
seen, sar_seen, checked = set(),set(),0
for pb,fb,tb in zip(pm['batches'],fm['batches'],tm['batches'],strict=True):
    for name,digest in tb['sha256'].items(): assert sha256(targets/tb['directory']/name)==digest
    info = json.loads((prepared/pb['directory']/'batch.json').read_text())
    fi = json.loads((features/fb['directory']/'batch.json').read_text())
    ti = json.loads((targets/tb['directory']/'batch.json').read_text())
    assert info['samples'] == fi['samples'] == ti['samples']
    maps = torch.load(prepared/pb['directory']/'reference_maps.pt',weights_only=True).numpy()
    values = {k:v.numpy() for k,v in torch.load(targets/tb['directory']/'targets.pt',weights_only=True).items()}
    for n,sample in enumerate(info['samples']):
        patch,split = sample['patch_id'],sample['split']
        assert patch not in seen and sample['s1_name'] not in sar_seen
        seen.add(patch); sar_seen.add(sample['s1_name'])
        source,transform,crs,area_bounds = original[patch]
        np.testing.assert_array_equal(source,maps[n])
        np.testing.assert_allclose(tuple(transform)[:6],sample['transform'],atol=0,rtol=0)
        np.testing.assert_allclose(area_bounds,sample['bounds'],atol=0,rtol=0)
        assert crs == rasterio.crs.CRS.from_user_input(sample['crs'])
        assert crs.is_projected and crs.linear_units_factor[1] == 1
        stats[split]['areas'] += 1
        for row in range(15):
            for col in range(15):
                token=row*15+col
                codes,counts=np.unique(source[row*8:row*8+8,col*8:col*8+8],return_counts=True)
                expected=np.zeros(19,dtype=np.int64); unknown=0
                for code,count in zip(codes,counts):
                    if int(code) in lookup: expected[lookup[int(code)]] += count
                    else:
                        assert int(code) in excluded
                        unknown += int(count)
                assert expected.sum()+unknown == 64
                np.testing.assert_array_equal(values['class_counts'][n,token],expected)
                np.testing.assert_array_equal(values['class_fractions'][n,token],expected/64)
                assert values['unlabeled_counts'][n,token] == unknown
                assert values['unlabeled_fraction'][n,token] == unknown/64
                assert values['fully_labeled_mask'][n,token] == (unknown==0)
                assert values['has_labels_mask'][n,token] == (unknown<64)
                ground=bounds(Window(col*8,row*8,8,8),transform)
                np.testing.assert_allclose(values['bounds'][n,token],ground,atol=0,rtol=0)
                assert ground[2]-ground[0] == ground[3]-ground[1] == 80
                stats[split]['fully_labeled' if unknown==0 else 'unlabeled' if unknown==64 else 'partially_labeled'] += 1
                if unknown==0:
                    class_tokens[split] += expected>0
                    class_pixels[split] += expected
                checked += 1
    print('Independently audited',len(seen),'areas /',checked,'tokens',flush=True)
assert len(seen)==1000 and checked==225000 and seen==set(original)
report={'stages':[3,4],'passed':True,'areas_checked':len(seen),'tokens_checked':checked,
        'class_counts_checked':checked*19,'original_reference_pixels_checked':1000*120*120,
        'ground_footprint_metres':[80,80],'token_order':'row-major',
        'fraction_denominator':64,'unlabeled_area_renormalized':False,
        'target_manifest_sha256':sha256(targets/'manifest.json'),
        'split_statistics':{s:dict(v) for s,v in stats.items()},
        'eligible_class_token_support':{s:v.tolist() for s,v in class_tokens.items()},
        'eligible_class_pixel_support':{s:v.tolist() for s,v in class_pixels.items()},
        'label_limitation':'Alignment verifies supplied CLC reference maps; it does not establish independent ground-truth accuracy at 80 metres.'}
(P/'stage-3-4-report.json').write_text(json.dumps(report,indent=2)+'\n')
del original,maps,values
print('STAGES 3–4 COMPLETE:',json.dumps(report),flush=True)


## 5. Train on training areas and choose the epoch using validation

In [ ]:
# Stage 5: train on train areas and choose an epoch using validation areas.
import json
import shutil
import tempfile
from pathlib import Path

import torch

from satquery.evaluation import load_splits, predict_rows
from satquery.prediction import save_head, load_head
from satquery.preprocessing import sha256
from satquery.validation_training import fit_with_validation

P=Path('/content/drive/MyDrive/SatQuery/pipeline-1000')
audit=json.loads((P/'stage-3-4-report.json').read_text())
assert audit['passed'] and audit['tokens_checked']==225000
assert sha256(P/'targets'/'manifest.json')==audit['target_manifest_sha256']
torch.set_num_threads(2)
splits, provenance=load_splits(P/'features',P/'targets',progress=lambda s:print(s,flush=True))
assert provenance['input_sample_splits']=={'train':600,'validation':200,'test':200}
assert all(not(set(splits[a].patch_ids)&set(splits[b].patch_ids)) for a,b in [('train','validation'),('train','test'),('validation','test')])
print('Eligible token counts:',provenance['eligible_tokens_by_split'],flush=True)
output=P/'model'
if not output.exists():
    model,fit=fit_with_validation(splits['train'],splits['validation'],device='cuda' if torch.cuda.is_available() else 'cpu',
                                  max_epochs=60,patience=10,learning_rate=.001,batch_size=1024,seed=17,
                                  progress=lambda s:print(s,flush=True))
    metadata={**provenance,**{k:v for k,v in fit.items() if k!='history'},
              'training_mode':'train_split_only','evaluation_scope':'validation_selection_only',
              'loss':'unweighted_soft_target_cross_entropy','torch_version':str(torch.__version__)}
    temporary=Path(tempfile.mkdtemp(prefix='.model-',dir=P))
    try:
        save_head(temporary/'head.pt',model,metadata)
        restored,_=load_head(temporary/'head.pt')
        torch.testing.assert_close(predict_rows(restored,splits['validation'].x[:256]),
                                   predict_rows(model,splits['validation'].x[:256]),rtol=0,atol=0)
        report={**metadata,'history':fit['history'],'checkpoint_sha256':sha256(temporary/'head.pt'),
                'training_mean_class_fractions':splits['train'].y.mean(0).tolist()}
        (temporary/'training.json').write_text(json.dumps(report,indent=2,allow_nan=False)+'\n')
        temporary.rename(output)
    except BaseException:
        shutil.rmtree(temporary);raise
else:
    report=json.loads((output/'training.json').read_text())
    assert report['feature_manifest_sha256']==provenance['feature_manifest_sha256']
    assert report['target_manifest_sha256']==provenance['target_manifest_sha256']
    assert report['checkpoint_sha256']==sha256(output/'head.pt')
stage5={'stage':5,'passed':True,'selected_epoch':report['selected_epoch'],
        'epochs_run':len(report['history']),'best_validation_loss':report['best_validation_loss'],
        'checkpoint_sha256':report['checkpoint_sha256'],'train_areas':600,'validation_areas':200,
        'test_used_for_selection':False,'eligible_tokens_by_split':provenance['eligible_tokens_by_split']}
(P/'stage-5-report.json').write_text(json.dumps(stage5,indent=2)+'\n')
if torch.cuda.is_available():torch.cuda.empty_cache()
print('STAGE 5 COMPLETE:',json.dumps(stage5),flush=True)


## 6–7. Test the frozen model, calculate class metrics and verify inference

In [ ]:
# Stages 6–7: one final held-out test evaluation and per-class historical reliability.
import csv
import json
from pathlib import Path
import numpy as np
import torch
from satquery.evaluation import load_splits, predict_rows, coverage_metrics
from satquery.prediction import load_head
from satquery.preprocessing import sha256
from satquery.prediction_data import class_schema
from satquery.predict_cover import export_predictions

P=Path('/content/drive/MyDrive/SatQuery/pipeline-1000')
previous=json.loads((P/'stage-5-report.json').read_text())
assert previous['passed'] and sha256(P/'model'/'head.pt')==previous['checkpoint_sha256']
model,meta=load_head(P/'model'/'head.pt')
assert meta['feature_manifest_sha256']==sha256(P/'features'/'manifest.json')
assert meta['target_manifest_sha256']==sha256(P/'targets'/'manifest.json')
if 'splits' not in globals():
    splits,provenance=load_splits(P/'features',P/'targets',progress=lambda s:print(s,flush=True))
assert provenance['feature_manifest_sha256']==meta['feature_manifest_sha256']
assert provenance['target_manifest_sha256']==meta['target_manifest_sha256']
assert splits['test'].split=='test' and len(splits['test'].patch_ids)==200
output=P/'evaluation'
output.mkdir(exist_ok=True)
report_path=output/'test-report.json'
if not report_path.exists():
    test=splits['test']
    pred=predict_rows(model,test.x)
    metrics=coverage_metrics(pred.numpy(),test.y.numpy(),test.area_ids,bootstrap_repeats=1000,seed=17)
    train_report=json.loads((P/'model'/'training.json').read_text())
    baseline=np.broadcast_to(np.array(train_report['training_mean_class_fractions']),test.y.shape)
    baseline_metrics=coverage_metrics(baseline,test.y.numpy(),test.area_ids,bootstrap_repeats=1000,seed=17)
    report={'format_version':1,'checkpoint_sha256':sha256(P/'model'/'head.pt'),
            'feature_manifest_sha256':meta['feature_manifest_sha256'],'target_manifest_sha256':meta['target_manifest_sha256'],
            'evaluation_scope':'held_out_test_areas','official_test_areas':200,
            'eligible_tokens_only':True,'selected_epoch':meta['selected_epoch'],
            'metrics':metrics,'training_mean_baseline':baseline_metrics}
    prediction_path=output/'test-predictions.pt'
    pending=output/'.test-predictions.pt.tmp'
    torch.save({'predicted_fractions':pred,'true_fractions':test.y,'area_ids':test.area_ids.tolist()},pending)
    restored=torch.load(pending,weights_only=True)
    assert torch.equal(restored['predicted_fractions'],pred)
    assert torch.equal(restored['true_fractions'],test.y) and restored['area_ids']==test.area_ids.tolist()
    pending.replace(prediction_path)
    report['test_predictions_sha256']=sha256(prediction_path)
    pending_report=output/'.test-report.json.tmp'
    pending_report.write_text(json.dumps(report,indent=2,allow_nan=False)+'\n')
    pending_report.replace(report_path)
else:
    report=json.loads(report_path.read_text())
    assert report['checkpoint_sha256']==sha256(P/'model'/'head.pt')
    metrics=report['metrics']
assert report['test_predictions_sha256']==sha256(output/'test-predictions.pt')
saved=torch.load(output/'test-predictions.pt',weights_only=True)
assert saved['area_ids']==splits['test'].area_ids.tolist()
assert torch.equal(saved['true_fractions'],splits['test'].y)
assert saved['predicted_fractions'].shape==splits['test'].y.shape and torch.isfinite(saved['predicted_fractions']).all()
del saved
reliability={'format_version':1,'checkpoint_sha256':report['checkpoint_sha256'],
             'feature_contract':meta['feature_contract'],'classes':class_schema(),
             'evaluation_scope':'held_out_test_areas','prediction_confidence_available':False,
             'test_report_sha256':sha256(report_path),'class_metrics':metrics['classes'],
             'note':'Show coverage prediction separately from historical test MAE, present-class MAE and area support. These are not calibrated probabilities or prediction intervals.'}
(output/'reliability.json').write_text(json.dumps(reliability,indent=2,allow_nan=False)+'\n')
fields=['index','name','mae_pp','rmse_pp','bias_pp','present_mae_pp','present_tokens','present_areas',
        'within_5pp_fraction','present_within_5pp_fraction','dominant_precision','dominant_recall','dominant_f1','support_status']
with (output/'per-class-metrics.csv').open('w',newline='') as f:
    writer=csv.DictWriter(f,fieldnames=fields,extrasaction='ignore');writer.writeheader();writer.writerows(metrics['classes'])
print('HELD-OUT TEST:',json.dumps({k:v for k,v in metrics.items() if k not in ('classes','dominant_confusion_matrix')}),flush=True)
print('BASELINE MAE:',report['training_mean_baseline']['coverage_mae_pp'],flush=True)
for c in metrics['classes']:
    print(c['name'],': MAE',round(c['mae_pp'],2),'pp; present MAE',None if c['present_mae_pp'] is None else round(c['present_mae_pp'],2),'pp; areas',c['present_areas'],flush=True)
# Exercise the inference module and attach the matching reliability artifact.
if not (P/'predictions').exists():
    export_predictions(P/'features',P/'model'/'head.pt',P/'predictions',batch_size=2048,
                       reliability=output/'reliability.json',progress=lambda s:print(s,flush=True))
validation=json.loads((P/'predictions'/'validation.json').read_text())
assert validation['passed'] and validation['manifest_sha256']==sha256(P/'predictions'/'manifest.json')
pm=json.loads((P/'predictions'/'manifest.json').read_text())
assert pm['checkpoint_sha256']==report['checkpoint_sha256']
for batch in pm['batches']:
    for name,digest in batch['sha256'].items(): assert sha256(P/'predictions'/batch['directory']/name)==digest
assert pm['historical_class_reliability']['sha256']==sha256(P/'predictions'/'reliability.json')
stage={'stages':[6,7],'passed':True,'test_area_count':200,'eligible_test_tokens':metrics['token_count'],
       'test_coverage_mae_pp':metrics['coverage_mae_pp'],'baseline_coverage_mae_pp':report['training_mean_baseline']['coverage_mae_pp'],
       'test_dominant_accuracy':metrics['dominant_class_accuracy'],'class_count':19,
       'test_report_sha256':sha256(report_path),'per_class_csv':str(output/'per-class-metrics.csv'),
       'reliability_report':str(output/'reliability.json'),'inference_areas_verified':pm['sample_count']}
(P/'stage-6-7-report.json').write_text(json.dumps(stage,indent=2,allow_nan=False)+'\n')
print('STAGES 6–7 COMPLETE:',json.dumps(stage),flush=True)


## Reading the results
`evaluation/per-class-metrics.csv` contains coverage MAE/RMSE/bias in percentage points, errors restricted to blocks where a class occurs, support, and dominant-class precision/recall/F1. Low overall error for an absent class does not establish reliability.

`evaluation/reliability.json` is bound to the exact saved model. The inference module copies it into its export. Display these historical errors separately from predicted cover percentage. No calibrated per-prediction confidence is claimed.

The test set has now been used for this baseline. Select later features and models on validation data; reserve a fresh test set for the eventual final comparison.
